In [ ]:
import pyarrow.parquet as pq
import polars as pl
import numpy as np
from tqdm.auto import tqdm
import gc

In [ ]:
print("Đang nạp tập Test và tạo danh sách đối chiếu...")
df_test = pl.read_parquet(TEST_PATH)
truth_df = df_test.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('true_items'))
n_valid = truth_df.height

print(f"Số lượng người dùng hợp lệ trong tập Test: {n_valid:,}")
print("Đang tính toán chỉ số bằng luồng dữ liệu (Streaming Evaluation)...")

FINAL_TOP100_PATH = '/kaggle/input/datasets/b22dckh072/feature-engineering/top100_final_recommendations.parquet'
pf = pq.ParquetFile(FINAL_TOP100_PATH)

In [ ]:
# Lấy 5 triệu dòng mỗi nhịp để đạt tốc độ tối đa mà không tốn RAM
reader = pf.iter_batches(batch_size=5000000, columns=['mapped_user_id', 'mapped_item_id'])

hr_sum = 0
ndcg_sum = 0.0
buffer_df = pl.DataFrame()

for batch in tqdm(reader, desc="Evaluating Chunks"):
    chunk = pl.from_arrow(batch)
    
    # Nối phần cắt dở từ chunk trước
    if buffer_df.height > 0:
        chunk = pl.concat([buffer_df, chunk])
        
    last_user = chunk.get_column('mapped_user_id')[-1]
    
    # Tách những user đã có đầy đủ danh sách ứng viên
    completed = chunk.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = chunk.filter(pl.col('mapped_user_id') == last_user)
    
    if completed.height > 0:
        # Đánh số thứ tự dựa trên vị trí hiện tại (vì file đã được sort từ trước)
        completed = completed.with_columns(
            pl.int_range(1, pl.len() + 1).over('mapped_user_id').alias('rank')
        )
        
        # Chỉ xét Top 10 để đánh giá HR@10 và NDCG@10
        top10 = completed.filter(pl.col('rank') <= 10)
        
        # Kết nối với danh sách mua thật trong tập Test
        eval_df = top10.join(truth_df, on='mapped_user_id', how='inner')
        
        # Kiểm tra trúng đích (Hit)
        hits = eval_df.filter(pl.col('mapped_item_id').is_in(pl.col('true_items')))
        
        if hits.height > 0:
            # Đếm số User có ít nhất 1 Hit để cộng vào Hit Rate
            hr_sum += hits.select('mapped_user_id').n_unique()
            
            # Tính NDCG: 1 / log2(rank + 1) và cộng dồn
            ndcg = hits.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
            ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()
            
        del top10, eval_df, hits
        
    del chunk, completed
    gc.collect()

# Xử lý đoạn buffer của User cuối cùng khi kết thúc vòng lặp
if buffer_df.height > 0:
    buffer_df = buffer_df.with_columns(
        pl.int_range(1, pl.len() + 1).over('mapped_user_id').alias('rank')
    )
    top10 = buffer_df.filter(pl.col('rank') <= 10)
    eval_df = top10.join(truth_df, on='mapped_user_id', how='inner')
    hits = eval_df.filter(pl.col('mapped_item_id').is_in(pl.col('true_items')))
    
    if hits.height > 0:
        hr_sum += hits.select('mapped_user_id').n_unique()
        ndcg = hits.with_columns((1.0 / np.log2(pl.col('rank') + 1)).alias('ndcg_val'))
        ndcg_sum += ndcg.select(pl.col('ndcg_val').sum()).item()

# Tính toán điểm số cuối cùng
HR10 = hr_sum / n_valid
NDCG10 = ndcg_sum / n_valid

print("=======================================")
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH XGBOOST")
print(f"Tập người dùng hợp lệ: {n_valid:,}")
print(f"Hit Rate @ 10 (HR@10): {HR10:.4f}")
print(f"NDCG @ 10:             {NDCG10:.4f}")
print("=======================================")